In [ ]:
Goal:

Train a model to understand language and learn general knowledge from massive amounts of text.

In [ ]:
Key Points:

1. Training Data:
LLMs are trained on trillions of tokens (words, subwords, or bytes).
Sources: books, Wikipedia, news, web pages, forums, code, etc.

2. Objective Function:
Most LLMs (like GPT) use next-word prediction (also called causal language modeling).
Given a sequence of words, predict the next word/token.

3. Learning:
The model learns linguistic structure: grammar, syntax, common phrases.
It also picks up general knowledge: facts, relationships, reasoning patterns.
This is unsupervised/self-supervised learning—no human labels are required.

4. Architecture:
Pre-training typically uses transformer-based architectures.
Large models have billions of parameters to capture complex patterns.

5. Training Process:
Input text is tokenized (subword/byte-level tokens).
Tokens are passed through the model to predict the next token.
Loss (usually cross-entropy) is computed, and weights are updated.
Repeat across billions of sequences until convergence.

6. Result:
The model becomes a general-purpose language model.
It can generate coherent text, answer questions, summarize, translate, etc.
Later, it can be fine-tuned for specific tasks (e.g., classification, summarization).


In [ ]:
Large Text Corpus → Tokenization → Transformer Model → Next-Word Prediction → Loss → Backpropagation → Updated Model

## Pre-training is general: it is not task-specific.
## The model learns structure and knowledge from data itself.

In [1]:
import re
import html
import string

# --- Cleaning functions ---
def clean_text(text):
    if not isinstance(text, str):
        return text
    text = re.sub(r"<[^>]+>", "", text)       # remove HTML tags
    text = html.unescape(text)                 # decode HTML entities
    text = re.sub(r"\\+", " ", text)          # remove literal backslashes
    text = re.sub(r"[\n\r\t]+", " ", text)    # replace escaped characters with space
    text = re.sub(r"\s+", " ", text)          # collapse multiple spaces
    return text.strip()

def normalize_text(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    return text.strip()

def filter_gibberish(text, min_words=5):
    words = text.split()
    return len(words) > min_words

def is_non_toxic(text):
    # Placeholder: keep all for now
    return True

# --- Full preprocessing pipeline ---
def preprocess_texts(texts):
    cleaned_texts = []
    for text in texts:
        text = clean_text(text)
        text = normalize_text(text)
        if filter_gibberish(text) and is_non_toxic(text):
            cleaned_texts.append(text)
    # Deduplicate
    unique_texts = list(dict.fromkeys(cleaned_texts))
    return unique_texts


In [2]:
# Example raw corpus
raw_corpus = [
    "Artificial intelligence is the future of technology.",
    "Machine learning helps computers learn from data.",
    "Transformers are state of the art in NLP.",
    "Neural networks can model complex patterns.",
    "Deep learning is a subset of machine learning."
]

# --- Preprocess the text ---
corpus = preprocess_texts(raw_corpus)
print("Preprocessed corpus:", corpus)

# Flatten corpus into a single string (character-level)
flat_corpus = " ".join(corpus)

# -----------------------------
# Mini GPT training code
# -----------------------------
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

# Build vocabulary
all_text = flat_corpus.split()
vocab = list(set(all_text))
stoi = {ch:i for i,ch in enumerate(vocab)}
itos = {i:ch for ch,i in stoi.items()}
vocab_size = len(vocab)

# Dataset for next-token prediction
class TinyDataset(Dataset):
    def __init__(self, text, block_size=3):
        self.block_size = block_size
        self.data = [stoi[ch] for ch in text]
    def __len__(self):
        return len(self.data) - self.block_size
    def __getitem__(self, idx):
        x = self.data[idx:idx+self.block_size]
        y = self.data[idx+1:idx+self.block_size+1]
        return torch.tensor(x), torch.tensor(y)

dataset = TinyDataset(all_text)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

# Tiny GPT-style model
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, n_embd=32):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, n_embd)
        self.transformer = nn.Transformer(
            d_model=n_embd, nhead=2, num_encoder_layers=1, num_decoder_layers=1
        )
        self.fc = nn.Linear(n_embd, vocab_size)

    def forward(self, x):
        x = self.embed(x)
        x = x.permute(1,0,2)
        x = self.transformer(x, x)
        x = x.permute(1,0,2)
        return self.fc(x)

model = MiniGPT(vocab_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Training loop
for epoch in range(50):
    total_loss = 0
    for x, y in dataloader:
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits.view(-1, vocab_size), y.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

# Generate text
def generate(model, start_char, length=20):
    model.eval()
    idx = torch.tensor([stoi[start_char]]).unsqueeze(0)
    output = [start_char]
    for _ in range(length):
        with torch.no_grad():
            logits = model(idx)
            next_idx = torch.argmax(logits[0,-1]).item()
            next_char = itos[next_idx]
            output.append(next_char)
            idx = torch.cat([idx, torch.tensor([[next_idx]])], dim=1)
    return " ".join(output)

print("Generated text:")
print(generate(model, start_char=flat_corpus.split()[0]))


Preprocessed corpus: ['artificial intelligence is the future of technology', 'machine learning helps computers learn from data', 'transformers are state of the art in nlp', 'neural networks can model complex patterns', 'deep learning is a subset of machine learning']


/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


Epoch 10, Loss: 11.6380
Epoch 20, Loss: 3.1961
Epoch 30, Loss: 2.0700
Epoch 40, Loss: 1.7792
Epoch 50, Loss: 0.8778
Generated text:
artificial intelligence is the future of technology machine learning helps computers learn from data transformers are state of technology machine learning


In [5]:
# Mini GPT character-level generation
prompt = "artificial"  # need to use lowercase, since we normalized text
generated_text = generate(model, start_char=prompt[0], length=50)
print(generated_text)


a subset of machine learning helps computers learn from data transformers are state of machine learning helps computers learn from data transformers are state of machine learning helps computers learn from data transformers are state of machine learning helps computers learn from data transformers are state of machine learning helps computers


# Input from different sources

In [6]:
from datasets import load_dataset
import re, html, string

# Load a small subset for demonstration
dataset = load_dataset("ag_news", split="train[:100]")  # take 100 examples

# --- Cleaning functions ---
def clean_text(text):
    text = re.sub(r"<[^>]+>", "", text)
    text = html.unescape(text)
    text = re.sub(r"\\+", " ", text)
    text = re.sub(r"[\n\r\t]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def normalize_text(text):
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def filter_gibberish(text, min_words=5):
    return len(text.split()) > min_words

# Preprocess dataset
cleaned_texts = []
for t in dataset["text"]:
    t = clean_text(t)
    t = normalize_text(t)
    if filter_gibberish(t):
        cleaned_texts.append(t)

# Deduplicate
unique_texts = list(dict.fromkeys(cleaned_texts))
print(f"Original size: {len(dataset)}, After cleaning & dedup: {len(unique_texts)}")

# Flatten into a single list of words
flat_corpus = " ".join(unique_texts).split()


Original size: 100, After cleaning & dedup: 100


In [7]:
# load and Preprocess

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

# Vocabulary
vocab = list(dict.fromkeys(flat_corpus))
stoi = {w: i for i, w in enumerate(vocab)}
itos = {i: w for w, i in stoi.items()}
vocab_size = len(vocab)

# Dataset
class WordDataset(Dataset):
    def __init__(self, words, block_size=5):
        self.block_size = block_size
        self.data = [stoi[w] for w in words]

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = self.data[idx:idx+self.block_size]
        y = self.data[idx+1:idx+self.block_size+1]
        return torch.tensor(x), torch.tensor(y)

dataset = WordDataset(flat_corpus)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

# Mini GPT model
class WordGPT(nn.Module):
    def __init__(self, vocab_size, n_embd=64):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, n_embd)
        self.transformer = nn.Transformer(
            d_model=n_embd, nhead=2, num_encoder_layers=1, num_decoder_layers=1
        )
        self.fc = nn.Linear(n_embd, vocab_size)

    def forward(self, x):
        x = self.embed(x)
        x = x.permute(1,0,2)
        x = self.transformer(x, x)
        x = x.permute(1,0,2)
        return self.fc(x)

model = WordGPT(vocab_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)


/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [8]:
# Train mini GPT

epochs = 5 

for epoch in range(epochs):
    total_loss = 0
    for x, y in dataloader:
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits.view(-1, vocab_size), y.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")


Epoch 5, Loss: 6054.4871


In [16]:
def generate_sampling(model, start_prompt, max_words=20, top_k=5):
    model.eval()
    words = start_prompt.lower().split()
    idxs = [stoi[w] for w in words if w in stoi]
    idx_tensor = torch.tensor([idxs])
    output_words = words.copy()

    for _ in range(max_words):
        with torch.no_grad():
            logits = model(idx_tensor)[0, -1]  # logits for last word
            probs = torch.softmax(logits, dim=-1)
            top_probs, top_idx = torch.topk(probs, top_k)
            next_idx = top_idx[torch.multinomial(top_probs, 1)].item()
            next_word = itos[next_idx]
            output_words.append(next_word)
            idx_tensor = torch.cat([idx_tensor, torch.tensor([[next_idx]])], dim=1)
    return " ".join(output_words)

generated = generate_sampling(model, "Artificial intelligence is", max_words=5, top_k=10)
print(generated)


artificial intelligence is to and and to a


In [ ]:
# From hugging face

from transformers import GPT2Tokenizer, GPT2LMHeadModel

# Load pre-trained model & tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("distilgpt2")
model = GPT2LMHeadModel.from_pretrained("distilgpt2")

# Prepare prompt
prompt = "Artificial Intelligence is"
inputs = tokenizer(prompt, return_tensors="pt")

# Generate text
outputs = model.generate(
    **inputs,
    max_length=50,
    do_sample=True,      # for variability
    top_k=50,            # top-k sampling
    top_p=0.95,          # nucleus sampling
    temperature=0.8
)

# Decode generated tokens
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text)


# do_sample=True → allows random sampling instead of greedy decoding.
# top_k and top_p → control diversity of the generated text.
# temperature → lower values → more conservative text, higher → more creative.

In [ ]:
# This is a Decoder-Only model

In [ ]:
## Decoder-Only Models

Architecture: Only the decoder stack of a Transformer (like GPT).
Training Objective: Next-word prediction / causal language modeling. The model predicts the next token given all previous tokens.
Use Case: Text generation, completion, code generation, chatbots..

Input:  The cat sat on
Decoder → Predicts next token "the"
Next input: "The cat sat on the" → Predicts "mat"


In [ ]:
## Encoder-Only Models

Architecture: Only the encoder stack of a Transformer (like BERT).
Training Objective: Typically masked language modeling (MLM). Random words in the input are masked, and the model learns to predict them.
Use Case: Great for understanding text (classification, NER, sentiment analysis, embeddings).
Example Models: BERT, RoBERTa, DistilBERT.


Input:  The cat sat on the [MASK] .
Encoder → Outputs contextual embeddings → Predict "mat"

In [ ]:
## Encoder-Decoder (Seq2Seq) Models

Architecture: Uses both encoder and decoder stacks. Encoder encodes the input; decoder generates output.
Training Objective: Typically sequence-to-sequence prediction, e.g., translation, summarization.
Use Case: Text-to-text tasks like translation, summarization, question-answering.

Example Models: T5, BART, MarianMT.